# µhIR Middle-end Cookbook

This guided tour follows a small fixed-trip kernel from canonical µIR into µhLS's sequencing-oriented µhIR. It then applies the built-in graph-optimization pipeline and inspects the structural facts that later HLS stages consume.

This is a guided showcase: every example is complete and intended for experimentation.

## Setup

Run this cell first. It locates the repository root, adds src/ to Python's import path, and imports the µIR and µhIR APIs used below.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "uhls").exists():
    repo_root = repo_root.parent

if not (repo_root / "src" / "uhls").exists():
    raise RuntimeError("could not locate repo root containing src/uhls")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from IPython.display import display

from uhls.frontend import lower_source_to_uir
from graphviz import Source
from uhls.middleend.passes.opt import (
    CSEPass,
    ConstPropPass,
    CopyPropPass,
    DCEPass,
    InlineCallsPass,
    PruneFunctionsPass,
)
from uhls.middleend.passes.util import PassContext, PassManager
from uhls.middleend.uir import format_module, verify_module
from uhls.backend.hls import lower_module_to_seq
from uhls.backend.hls.uhir import format_uhir, parse_uhir, to_dot
from uhls.backend.hls.uhir.gopt import create_builtin_gopt_pass, run_gopt_passes


def show_design(design, max_lines=100):
    lines = format_uhir(design).splitlines()
    print("\n".join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more lines)")


def show_dot(title, dot):
    print(title)
    display(Source(dot))


print(f"repo_root = {repo_root}")

## 1. Begin with canonical µIR

The frontend produces typed, block-based SSA. A short cleanup pipeline removes copies and dead operations before the representation crosses into the HLS-oriented middle end.

The example deliberately contains a fixed-trip loop. That fact is visible in µIR control flow, but it has not yet become an explicit scheduling property.

In [ ]:
KERNEL_C_SOURCE = """
int32_t kernel(int32_t a, int32_t b, int32_t c) {
    int32_t acc = a;
    int32_t i = 0;
    for (i = 0; i < 4; i = i + 1) {
        acc = acc + b;
    }
    return acc * c;
}
"""

uir_module = lower_source_to_uir(KERNEL_C_SOURCE)
verify_module(uir_module)

uir_pipeline = [
    InlineCallsPass(),
    PruneFunctionsPass(),
    DCEPass(),
    CSEPass(),
    CopyPropPass(),
    ConstPropPass(),
    DCEPass(),
]
canonical_uir = PassManager(uir_pipeline).run(deepcopy(uir_module), PassContext())
verify_module(canonical_uir)

print("C input:\n")
print(KERNEL_C_SOURCE.strip())
print("\nCanonical µIR:\n")
print(format_module(canonical_uir))

## 2. Lower µIR to sequencing µhIR

µIR answers software questions: which basic block runs next, which SSA value reaches a use, and which operations define program behavior.

The seq stage of µhIR recasts that program as regions, operation nodes, and explicit dependency edges. It is still technology-independent—allocation, cycle numbers, and concrete functional-unit instances have not been chosen.

In [ ]:
seq_design = lower_module_to_seq(canonical_uir, top="kernel")
seq_text = format_uhir(seq_design)

# Textual µhIR is a real interchange form: parse it back and compare.
seq_roundtrip = parse_uhir(seq_text)
assert format_uhir(seq_roundtrip) == seq_text

print("seq-stage µhIR:\n")
show_design(seq_design)
show_dot("Raw sequencing graph", to_dot(seq_design, compact=True))

### Inspect the graph model

A design contains regions. Regions contain operation nodes and dependency edges, and hierarchical nodes may refer to child regions. This makes control structure explicit without committing to clock cycles yet.

In [ ]:
for region in seq_design.regions:
    print(
        f"{region.id}: kind={region.kind}, parent={region.parent}, "
        f"nodes={len(region.nodes)}, edges={len(region.edges)}"
    )
    for node in region.nodes:
        role = node.attributes.get("role")
        detail = f", role={role}" if role else ""
        print(f"  {node.id:>4}: {node.opcode}{detail}")

## 3. Make static structure explicit

The graph pipeline performs four distinct jobs:

1. recognize loop-shaped control structure;
2. translate it to the loop dialect;
3. infer compile-time properties such as a fixed trip count; and
4. simplify the resulting static control graph.

Running one pass at a time makes the canonicalization boundary visible.

In [ ]:
gopt_pipeline_names = [
    "infer_loops",
    "translate_loop_dialect",
    "infer_static",
    "simplify_static_control",
]

gopt_snapshots = [("seq", deepcopy(seq_design))]
current = deepcopy(seq_design)
for pass_name in gopt_pipeline_names:
    current = run_gopt_passes(current, [create_builtin_gopt_pass(pass_name)])
    gopt_snapshots.append((pass_name, deepcopy(current)))

for stage_name, design in gopt_snapshots:
    loop_nodes = [
        node
        for region in design.regions
        for node in region.nodes
        if node.opcode == "loop"
    ]
    trip_counts = [node.attributes.get("static_trip_count") for node in loop_nodes]
    print(
        f"{stage_name:>25}: regions={len(design.regions):2}, "
        f"nodes={sum(len(region.nodes) for region in design.regions):2}, "
        f"loop_trip_counts={trip_counts}"
    )

static_seq_design = gopt_snapshots[-1][1]
static_loop_nodes = [
    node
    for region in static_seq_design.regions
    for node in region.nodes
    if node.opcode == "loop" and isinstance(node.attributes.get("static_trip_count"), int)
]
assert static_loop_nodes
assert static_loop_nodes[0].attributes["static_trip_count"] == 4

print("\nGraph-optimized µhIR:\n")
show_design(static_seq_design)
show_dot("Static sequencing graph", to_dot(static_seq_design, compact=True))

## 4. Why canonical forms matter

The source loop began as ordinary branches and SSA values. The optimized graph now carries the machine-checkable fact static_trip_count=4 on a loop node. Allocation and scheduling can consume that fact directly instead of rediscovering source-level intent.

The text form also remains inspectable and round-trippable, which is useful when debugging transformations or comparing pipeline stages.

In [ ]:
loop = static_loop_nodes[0]
print("loop node:", loop.id)
print("opcode:", loop.opcode)
print("static trip count:", loop.attributes["static_trip_count"])
print("attributes:")
for name, value in sorted(loop.attributes.items()):
    print(f"  {name} = {value}")

static_text = format_uhir(static_seq_design)
assert format_uhir(parse_uhir(static_text)) == static_text
print("\nRound-trip check: PASS")

## Takeaways

- Canonical µIR is a typed SSA control-flow representation.
- seq-stage µhIR is a hierarchy of operation and dependency graphs.
- Graph passes expose loop structure and static bounds as explicit attributes.
- The result is still independent of a concrete schedule or resource implementation.
- The backend cookbook continues with allocation, scheduling, binding, and resource maps.